In [ ]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
from langchain.messages import SystemMessage
import operator
from langchain.messages import ToolMessage
from typing import Literal
from langgraph.graph import StateGraph, START, END



In [2]:
model = init_chat_model("openai:gpt-4.1-nano", temperature=0)

In [3]:
@tool
def addition(a:float,b:float)->float:
    """to add the to variable a and b"""
    return a + b

@tool
def subtraction(a:float,b:float)->float:
    """to subtract the variable b from a"""
    return a - b

In [ ]:
tools=[addition,subtraction]
model_with_tools=model.bind_tools(tools)
model_tool_names={tool.name: tool for tool in tools}

In [ ]:
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls:int

In [ ]:
def llm_calls(state:dict):
    "LLM decides whether to call the LLM or not!"

    return {
        "messages":[
            model_with_tools.invoke(
                [ SystemMessage(
                        content="You are a helpful assistant tasked with performing arithmetic on a set of inputs."
                    )] + state["messages"]
            )
        ], "llm_calls": state.get("llm_calls", 0) + 1
    }

In [ ]:
def tool_node(state:dict):
    """Performs the tool calls"""

    result=[]
    for tool_calls in state["messages"].tool_calls:
        tool=model_tool_names[tool_calls["name"]]
        observation=tool.invoke(tool_calls["args"])
        result.append(ToolMessage(content=observation,tool_calls_id=tool_calls["id"]))
    return {"message":result}

In [ ]:
def should_continue(state:MessagesState) -> Literal[True,False]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""
    